In [ ]:
  from datasets import load_dataset
  import numpy as np
  from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
  from collections import Counter
  import torch
  import torch.nn as nn
  import torch.nn.functional as F
  from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
  from sklearn.metrics import classification_report, confusion_matrix

  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  print(f"Using device: {device}")

  # Load the dataset
  dataset = load_dataset("ailsntua/QEvasion")

  # Prepare labels
  labels = dataset['train'].unique('clarity_label')
  num_labels = len(labels)
  label2id = {label: i for i, label in enumerate(labels)}
  id2label = {i: label for i, label in enumerate(labels)}

  def add_labels(example):
      example['labels'] = label2id[example['clarity_label']]
      return example

  dataset = dataset.map(add_labels)
  dataset = dataset.remove_columns([
      col for col in dataset['train'].column_names if col not in ['question', 'interview_answer', 'labels']
  ])

  print("Dataset ready:")
  print(dataset)
  print(f"Labels mapped: {label2id}")


  # Calculate class weights for imbalanced data [citation:1][citation:4][citation:7]
  def get_class_weights(dataset, num_labels):
      label_counts = Counter(dataset["train"]["labels"])
      total_samples = len(dataset["train"])

      # Calculate class weights (inverse frequency) [citation:5]
      class_weights = []
      for i in range(num_labels):
          count = label_counts.get(i, 1)  # avoid division by zero
          weight = total_samples / (num_labels * count)
          class_weights.append(weight)

      # Move the tensor to the active device (GPU)
      return torch.tensor(
          class_weights, dtype=torch.float32, device=device
      )


  # Get class weights
  class_weights = get_class_weights(dataset, num_labels)
  print(f"Class weights: {class_weights}")
  print(f"Class weights device: {class_weights.device}")  # Verify it's on CUDA

  # Custom Trainer with Weighted Cross-Entropy Loss [citation:1][citation:4]
  class CustomTrainer(Trainer):
      """
      Custom trainer that uses Weighted Cross-Entropy Loss with class weights
      This subclass overrides the compute_loss method to use weighted cross entropy
      """

      def __init__(self, *args, class_weights=None, **kwargs):
          super().__init__(*args, **kwargs)
          self.class_weights = class_weights
          # Initialize CrossEntropyLoss with class weights [citation:1][citation:7]
          self.cross_entropy_loss = nn.CrossEntropyLoss(weight=class_weights)

      def compute_loss(
          self, model, inputs, return_outputs=False, num_items_in_batch=None
      ):
          # Extract labels and run model forward pass
          labels = inputs.get("labels")
          outputs = model(**inputs)
          logits = outputs.get("logits")

          # Compute weighted cross entropy loss [citation:1][citation:4]
          # PyTorch's CrossEntropyLoss automatically applies the weights during reduction [citation:1]
          loss = self.cross_entropy_loss(logits, labels)

          # Handle return_outputs as required by the Trainer
          return (loss, outputs) if return_outputs else loss

  # Tokenization and model setup
  model_checkpoint = "answerdotai/ModernBERT-large"
  tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

  def tokenize_function(examples):
      return tokenizer(
          examples['question'],
          examples['interview_answer'],
          truncation=True,
          padding="max_length",
          max_length=1680
      )

  tokenized_datasets = dataset.map(tokenize_function, batched=True)

  model = AutoModelForSequenceClassification.from_pretrained(
      model_checkpoint,
      num_labels=num_labels,
      id2label=id2label,
      label2id=label2id
  )

  def compute_metrics(p):
      predictions, labels = p
      predictions = np.argmax(predictions, axis=1)

      # Use macro averaging for balanced metrics across classes
      precision_macro = precision_score(labels, predictions, average='macro', zero_division=0)
      recall_macro = recall_score(labels, predictions, average='macro', zero_division=0)
      f1_macro = f1_score(labels, predictions, average='macro', zero_division=0)

      # Keep weighted for comparison
      precision_weighted = precision_score(labels, predictions, average='weighted', zero_division=0)
      recall_weighted = recall_score(labels, predictions, average='weighted', zero_division=0)
      f1_weighted = f1_score(labels, predictions, average='weighted', zero_division=0)

      acc = accuracy_score(labels, predictions)

      return {
          'accuracy': acc,
          'f1_macro': f1_macro,
          'f1_weighted': f1_weighted,
          'precision_macro': precision_macro,
          'precision_weighted': precision_weighted,
          'recall_macro': recall_macro,
          'recall_weighted': recall_weighted
      }

  # Training Arguments
  training_args = TrainingArguments(
      output_dir="ModernBERT_QEvasion_model",
      learning_rate=1e-5,
      per_device_train_batch_size=8,
      num_train_epochs=8,
      weight_decay=0.01,
      eval_strategy="epoch",
      save_strategy="epoch",
      load_best_model_at_end=True,
      metric_for_best_model="f1_macro",
      greater_is_better=True,
      push_to_hub=False,
      logging_steps=100,
      report_to="none",
      fp16=True,  # Enable mixed precision (reduces memory usage)
      gradient_checkpointing=True,
  )

  # Updated CustomTrainer instantiation with Weighted Cross-Entropy Loss
  trainer = CustomTrainer(
      model=model,
      args=training_args,
      train_dataset=tokenized_datasets["train"],
      eval_dataset=tokenized_datasets["test"],
      tokenizer=tokenizer,
      compute_metrics=compute_metrics,
      class_weights=class_weights,  # Pass the calculated class weights for CrossEntropyLoss
  )

  print(f"Using class weights: {class_weights}")
  print("Starting training with Weighted Cross-Entropy Loss...")
  trainer.train()

  print("Training completed!")

  # Final Evaluation
  test_results = trainer.evaluate()
  print("\n" + "="*60)
  print(f"FINAL TEST RESULTS (from best epoch: {trainer.state.best_model_checkpoint})")
  print("="*60)
  for key, value in test_results.items():
      if key not in ['epoch', 'eval_runtime', 'eval_samples_per_second', 'eval_steps_per_second']:
          print(f"{key}: {value:.4f}")

  # Optional: Get detailed predictions
  print("\nDetailed predictions analysis (from best model):")
  predictions = trainer.predict(tokenized_datasets["test"])
  predicted_labels = np.argmax(predictions.predictions, axis=1)
  true_labels = predictions.label_ids

  print("\nClassification Report:")
  print(classification_report(true_labels, predicted_labels,
                            target_names=[id2label[i] for i in range(num_labels)]))

  print("\nConfusion Matrix:")
  print(confusion_matrix(true_labels, predicted_labels))